In [ ]:
import numpy as np
import pandas as pd

def pose_calculation(
    csv_path: str,
    out_csv_path: str | None = None,
    ts_col: str = "neu_ts",
    use_dt_from_timestamps: bool = True,
    fps: float = 30.0,
    assume_sorted: bool = False,
) -> pd.DataFrame:
    """
    Load a CSV and compute:
      - ear midpoint (ear_mid_x, ear_mid_y)
      - head direction angle in radians (head_dir_rad)
      - angular velocity (ang_vel_rad_s, ang_vel_deg_s)

    Expected columns (matching your aligned_min output):
      - ear_L.x, ear_L.y, ear_R.x, ear_R.y, nose.x, nose.y
    Also expects a timestamp column (default: 'neu_ts') if use_dt_from_timestamps=True.

    Notes:
      - If any required pose value is NaN, the derived quantities become NaN automatically.
      - Angular velocity uses unwrap to avoid +/-pi jumps, then differentiates by time.
    """
    df = pd.read_csv(csv_path)

    required = ["ear_L.x", "ear_L.y", "ear_R.x", "ear_R.y", "nose.x", "nose.y"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    # --- timestamps / ordering ---
    if use_dt_from_timestamps:
        if ts_col not in df.columns:
            raise ValueError(f"use_dt_from_timestamps=True but ts_col='{ts_col}' not found in CSV.")
        df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")

        if not assume_sorted:
            df = df.sort_values(ts_col).reset_index(drop=True)

        # dt in seconds (first is NaN)
        dt = df[ts_col].diff().dt.total_seconds().to_numpy()
    else:
        if not assume_sorted:
            # keep original order; user is telling us dt is constant
            df = df.reset_index(drop=True)
        dt = np.full(len(df), 1.0 / float(fps), dtype=float)
        dt[0] = np.nan

    # --- ear midpoint (exactly like your main block) ---
    df["ear_mid_x"] = (df["ear_L.x"] + df["ear_R.x"]) / 2.0
    df["ear_mid_y"] = (df["ear_L.y"] + df["ear_R.y"]) / 2.0

    # --- head direction (nose - ear_mid) ---
    df["hd_dx"] = df["nose.x"] - df["ear_mid_x"]
    df["hd_dy"] = df["nose.y"] - df["ear_mid_y"]
    df["head_dir_rad"] = np.arctan2(df["hd_dy"], df["hd_dx"])

    # --- angular velocity ---
    # unwrap only on valid angles; keep NaNs where angle is NaN
    ang = df["head_dir_rad"].to_numpy(dtype=float)
    ang_unwrapped = np.full_like(ang, np.nan, dtype=float)

    valid = np.isfinite(ang)
    if valid.any():
        ang_unwrapped[valid] = np.unwrap(ang[valid])

    # differentiate: dtheta/dt
    dtheta = np.diff(ang_unwrapped)
    ang_vel = np.full(len(df), np.nan, dtype=float)

    # dt applies to the step from i-1 -> i, so use dt[1:]
    dt_step = dt[1:]
    good_steps = np.isfinite(dtheta) & np.isfinite(dt_step) & (dt_step > 0)

    ang_vel[1:][good_steps] = dtheta[good_steps] / dt_step[good_steps]

    df["ang_vel_rad_s"] = ang_vel
    df["ang_vel_deg_s"] = np.degrees(ang_vel)

    if out_csv_path is not None:
        df.to_csv(out_csv_path, index=False)

    return df

In [ ]:
import matplotlib.pyplot as plt

in_csv  = r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\1028_dataloader.csv"
out_csv = r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\1028_dataloader_HDcalc.csv"

df_test = pose_calculation(
    csv_path=in_csv,
    out_csv_path=out_csv,
    ts_col="neu_ts",              # IMPORTANT: this matches your file
    use_dt_from_timestamps=True,  # recommended
    fps=30.0
)

print(df_test.columns)
print(df_test[[
    "ear_mid_x",
    "ear_mid_y",
    "head_dir_rad",
    "ang_vel_rad_s"
]].head(10))

plt.figure()
plt.plot(df_test["head_dir_rad"])
plt.title("Head Direction (rad)")
plt.show()

plt.figure()
plt.plot(df_test["ang_vel_rad_s"])
plt.title("Angular Velocity (rad/s)")
plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_trajectory_over_arena(
    csv_path: str,
    video_path: str,
    x_col: str = "ear_mid_x",
    y_col: str = "ear_mid_y",
    ts_col: str = "neu_ts",
    use_in_arena: bool = True,
    arena_col: str = "in_arena",
    downsample: int = 1
):
    """
    Plots mouse trajectory over first frame of arena video.
    Colors points by timestamp progression.
    """

    # Load dataframe
    df = pd.read_csv(csv_path)

    if ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], utc=True)

    # Optional arena filtering
    if use_in_arena and arena_col in df.columns:
        df = df[df[arena_col] == True]

    # Remove NaNs
    df = df[np.isfinite(df[x_col]) & np.isfinite(df[y_col])]

    # Downsample for performance
    if downsample > 1:
        df = df.iloc[::downsample]

    # Load first video frame
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError("Could not read video frame.")

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Normalize time for coloring
    if ts_col in df.columns:
        t = df[ts_col].astype("int64") / 1e9
        t_norm = (t - t.min()) / (t.max() - t.min())
    else:
        t_norm = np.linspace(0, 1, len(df))

    # Plot
    plt.figure(figsize=(8, 8))
    plt.imshow(frame_rgb)
    sc = plt.scatter(
        df[x_col],
        df[y_col],
        c=t_norm,
        cmap="viridis",
        s=3
    )
    plt.colorbar(sc, label="Time progression")
    plt.title("Mouse Trajectory Over Arena")
    plt.gca().invert_yaxis()  # match image coordinates
    plt.show()

In [ ]:
plot_trajectory_over_arena(
    csv_path=r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\1028_dataloader_HDcalc.csv",
    video_path=r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\20251028_behcam.avi",
    x_col="ear_mid_x",
    y_col="ear_mid_y",
    ts_col="neu_ts",
    use_in_arena=True,
    downsample=5  
)

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_head_direction_over_arena(
    csv_path: str,
    video_path: str,
    x_col: str = "ear_mid_x",
    y_col: str = "ear_mid_y",
    angle_col: str = "head_dir_rad",
    ts_col: str = "neu_ts",
    use_in_arena: bool = True,
    arena_col: str = "in_arena",
    downsample: int = 30,          # ~1 arrow per second at 30 fps
    arrow_len: float = 30.0,       # pixels
    alpha: float = 0.7,
):
    """
    Overlay head-direction arrows on the arena background image.
    Arrows originate at (x_col, y_col) and point along angle_col (radians).
    Colors arrows by time progression (if ts_col exists).
    """

    df = pd.read_csv(csv_path)

    # optional timestamp parsing for coloring
    if ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors="coerce")

    # optional arena filter
    if use_in_arena and arena_col in df.columns:
        df = df[df[arena_col] == True]

    # keep only valid rows
    valid = (
        np.isfinite(df[x_col].to_numpy()) &
        np.isfinite(df[y_col].to_numpy()) &
        np.isfinite(df[angle_col].to_numpy())
    )
    df = df.loc[valid].copy()

    # downsample for readability
    if downsample > 1:
        df = df.iloc[::downsample].copy()

    # load first video frame
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f"Could not read first frame from: {video_path}")

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # time progression for color
    if ts_col in df.columns and df[ts_col].notna().any():
        t = df[ts_col].astype("int64") / 1e9
        t_norm = (t - t.min()) / (t.max() - t.min() + 1e-12)
    else:
        t_norm = np.linspace(0, 1, len(df))

    # arrow vectors (image coords: +x right, +y down)
    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()
    ang = df[angle_col].to_numpy()

    u = np.cos(ang) * arrow_len
    v = np.sin(ang) * arrow_len  # positive v goes downward in image coordinates

    plt.figure(figsize=(8, 8))
    plt.imshow(frame_rgb)

    # quiver: arrows; angles are in data coords
    q = plt.quiver(
        x, y, u, v, t_norm,
        angles="xy",
        scale_units="xy",
        scale=1,
        cmap="viridis",
        alpha=alpha,
        width=0.003
    )

    plt.colorbar(q, label="Time progression")
    plt.title("Head Direction Over Arena")
    plt.gca().invert_yaxis()  # keep the same coordinate convention as your trajectory plot
    plt.show()

In [ ]:
plot_head_direction_over_arena(
    csv_path=r"C:\Users\psych-aalab\Desktop\aligned_minimal_with_kinematics.csv",
    video_path=r"C:\Users\psych-aalab\Desktop\zenon_frametest\20251028\20251028_behcam.avi",
    downsample=10,     # try 10 for denser, 60 for sparser
    arrow_len=25
)